## Import and install dependencies

In [1]:
!pip install evaluate transformers torch accelerate sentencepiece tiktoken -q

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, BertForSequenceClassification


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)

cuda


In [3]:
BASE_MODEL_NAME = "prajjwal1/bert-tiny"

num_labels = 2
id2label = {0:'BR', 1:'PT'}
label2id = {'BR':0, 'PT':1}

# model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)
model = BertForSequenceClassification.from_pretrained(BASE_MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

In [ ]:
model.config

BertConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 128,
  "id2label": {
    "0": "BR",
    "1": "PT"
  },
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "is_decoder": false,
  "label2id": {
    "BR": 0,
    "PT": 1
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 2,
  "num_hidden_layers": 2,
  "pad_token_id": 0,
  "tie_word_embeddings": true,
  "transformers_version": "5.9.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

## Loading Dataset

In [5]:
from datasets import Dataset, DatasetDict, load_dataset

SUBSET_SIZE = 1000

pt_stream = load_dataset('bastao/VeraCruz_PT-BR', "Portugal (PT)", split='train', streaming=True)
ptbr_stream = load_dataset('bastao/VeraCruz_PT-BR', "Brazil (BR)", split='train', streaming=True)
other_stream = load_dataset('bastao/VeraCruz_PT-BR', "Other", split='train', streaming=True)

pt_subset = pt_stream.take(SUBSET_SIZE)
ptbr_subset = ptbr_stream.take(SUBSET_SIZE)
other_subset = other_stream.take(SUBSET_SIZE)

label_map = {"BR": 0, "PT": 1} # Adjust based on your actual classes

def stream_to_dataset(stream, label=None):
    dataset = Dataset.from_list(list(stream))
    if label is not None:
        dataset = dataset.add_column("label", [label] * len(dataset))
    return dataset

def preprocess_labels(example):
    # Replace 'label' with the actual name of your label column
    example["label"] = label_map[example["label"]]
    return example

VERA_CRUZ_DATASETS = DatasetDict({
    "ptbr": stream_to_dataset(ptbr_subset, label_map["BR"]),
    "pt": stream_to_dataset(pt_subset, label_map["PT"]),
    "other": stream_to_dataset(other_subset).map(preprocess_labels),
})

VERA_CRUZ_DATASETS

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    ptbr: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'label'],
        num_rows: 1000
    })
    pt: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'label'],
        num_rows: 1000
    })
    other: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'label', 'score'],
        num_rows: 1000
    })
})

In [ ]:
VERA_CRUZ_DATASETS['ptbr'][0]

{'text': 'Covid-19: Ramiro defende protocolo do Corinthians após polêmica | LANCE!\nCovid-19: Ramiro defende protocolo do Corinthians após polêmica\nO volante comentou sobre o assunto durante entrevista coletiva nesta tarde no CT, juntamente com o companheiro de equipe Gabriel, que falou sobre a final sem torcida\nUma polêmica que tomou conta nos últimos dias foi a recusa do Corinthians em realizar os testes de Covid-19 em seu elenco, antes das finais do Campeonato Paulista. Perguntado sobre o tema em entrevista coletiva juntamente com Gabriel, o volante Ramiro disse que essa questão não irá afetar a preparação da equipe.\n- Esse tipo de questão a gente deixa mais para o extracampo, extra-clube, temos que nos focar no nosso trabalho, ir a campo e fazer um grande jogo. Isso não vai interferir na nossa concentração para esses dois jogos. Estudamos muito a equipe deles, sabemos a forma que a gente tem que se portar para, pós segundo jogo, nos sagrarmos campeões – completou.\nO fato gerou 

## Tokenize

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

def preprocess(sample):
    return tokenizer(sample['text'], truncation=True)

# Trunacting max_length=512
TOKENIZED_VERA_CRUZ_DATASETS = VERA_CRUZ_DATASETS.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=512),
    batched=True
)

TOKENIZED_VERA_CRUZ_DATASETS

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.

In [ ]:
TOKENIZED_VERA_CRUZ_DATASETS["other"]

## Fine-tuning

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir='teste',
    per_device_train_batch_size=64,
    logging_strategy='epoch',
    num_train_epochs=3,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=TOKENIZED_VERA_CRUZ_DATASETS['other'],
    # eval_dataset=tokenized_imdb["test"],
    data_collator=data_collator,
    # compute_metrics=compute_metrics,
)


In [ ]:
trainer.train()

## Evaluation

In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return accuracy_score(labels, preds)

preds = trainer.predict(LABELED_DATASET)
compute_metrics(preds)